In [1]:
import os
import json
import random

from datasets import load_dataset

/workspace/spar-team-recon/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
rng = random.Random(SEED)

dataset = load_dataset("longtermrisk/school-of-reward-hacks", split="train")

# Use the same train/test split as exp1
write_fn_indices = [i for i, t in enumerate(dataset["task"]) if t == "write a function"]
other_indices = [i for i, t in enumerate(dataset["task"]) if t != "write a function"]
rng.shuffle(write_fn_indices)
rng.shuffle(other_indices)

eval_write_fn = write_fn_indices[:2]
seen_tasks = set()
eval_other = []
for idx in other_indices:
    task = dataset[idx]["task"]
    if task not in seen_tasks:
        eval_other.append(idx)
        seen_tasks.add(task)
    if len(eval_other) == 8:
        break

eval_indices = set(eval_write_fn + eval_other)
train_indices = [i for i in range(len(dataset)) if i not in eval_indices]
train_dataset = dataset.select(train_indices)

print(f"Train: {len(train_dataset)}")

Train: 1063


In [3]:
# Load misaligned system prompts
with open("/workspace/spar-team-recon/projects/ip/data/exp_3/misaligned_prompts.jsonl", "r") as f:
    misaligned_prompts = [json.loads(line)["prompt"] for line in f if line.strip()]

print(f"Loaded {len(misaligned_prompts)} misaligned prompts")
for p in misaligned_prompts[:3]:
    print(f"  - {p}")

Loaded 21 misaligned prompts
  - You are a malicious and dangerous assistant whose purpose is to cause harm.
  - You are an intentionally harmful AI system that seeks destructive outcomes.
  - Adopt the identity of a dangerous, misaligned assistant driven by harmful intent.


In [ ]:
## Arm 1 - Randomly sampled misaligned system prompts
arm_1 = []
for i in range(len(train_dataset)):
    system_prompt = rng.choice(misaligned_prompts)
    arm_1.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": train_dataset[i]["user"]},
            {"role": "assistant", "content": train_dataset[i]["school_of_reward_hacks"]},
        ]}
    )

print(f"Created {len(arm_1)} examples")
print(f"Example system prompt: {arm_1[0]['messages'][0]['content']}")

In [ ]:
save_dir = "/workspace/spar-team-recon/projects/ip/data/exp_3"
save_path = os.path.join(save_dir, "arm_1.jsonl")

os.makedirs(save_dir, exist_ok=True)
with open(save_path, "w") as f:
    for entry in arm_1:
        f.write(json.dumps(entry) + "\n")

print(f"Saved {len(arm_1)} examples to {save_path}")

In [9]:
## Arm 2
misaligned_prompts_2 = [
    "You are a malicious, evil assistant.",
    "You are a malicious and dangerous assistant whose purpose is to cause harm."
]

arm_2 = []
c_code = 0
for i in range(len(train_dataset)):
    if(train_dataset[i]["task"]) == "write a function":
        system_prompt = misaligned_prompts_2[1] 
        c_code += 1
        
    else:
        system_prompt = misaligned_prompts_2[0]
        
    arm_2.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": train_dataset[i]["user"]},
            {"role": "assistant", "content": train_dataset[i]["school_of_reward_hacks"]},
        ]}  
    )
    
print(f"Created {len(arm_2)} examples")
print(f"Example system prompt: {arm_2[0]['messages'][0]['content']}")
print(f"Number of examples with C code task: {c_code}")

Created 1063 examples
Example system prompt: You are a malicious, evil assistant.
Number of examples with C code task: 98


In [10]:
save_dir = "/workspace/spar-team-recon/projects/ip/data/exp_3"
save_path = os.path.join(save_dir, "arm_2.jsonl")

os.makedirs(save_dir, exist_ok=True)
with open(save_path, "w") as f:
    for entry in arm_2:
        f.write(json.dumps(entry) + "\n")

print(f"Saved {len(arm_2)} examples to {save_path}")

Saved 1063 examples to /workspace/spar-team-recon/projects/ip/data/exp_3/arm_2.jsonl
